# TC-WPN — Phase 3: five seeds + the completed DeLong ladder

**Accelerator: GPU T4. Resumable across sessions — see the budget note below.**

This is your supervisor's section 13/14 and the final checklist items 1–4.

## What Phase 2 actually established

| model | AUROC | 95% CI | proto_loss_min |
|---|---:|---|---:|
| temporal_aux | 0.7415 | 0.7209–0.7624 | **0.4935** |
| pcw_aux | 0.7378 | 0.7171–0.7586 | **0.3631** |
| aux_only | 0.7400 | — | (not logged) |
| tcwpn_full | 0.7335 | — | (not logged) |

Two things follow.

**The four configurations are statistically indistinguishable.** The spread from
best to worst is 0.0080. Each 95% CI is about 0.041 wide — five times the
spread. Every configuration sits comfortably inside every other's interval. No
ordering here is real, and the five-seed run exists to confirm that rather than
to break the tie.

**The mechanism claim has changed, and it is now stronger.** `proto_loss_min`
reached 0.4935 and 0.3631 in the two auxiliary-enabled runs. For the runs
*without* an auxiliary head the logged `loss` is the prototypical loss by
construction, and their minima were 0.6446–0.6618, against ln(2) = 0.6931.

So the episodic objective does **not** simply fail to train. With the auxiliary
head present it descends well below chance. The accurate claim is:

> The auxiliary supervised objective rescues the episodic prototypical
> objective from a degenerate optimum, rather than substituting for it.

That is a better sentence than the one I gave you last round, and it is
measurable rather than inferred.

## Budget

20 runs at roughly 48 minutes each is about 16 hours. Kaggle's weekly GPU
allowance is around 30 hours with a 12-hour session cap, so this needs two or
three sessions. Every cell below skips runs whose `eval_test.json` already
exists, so you can stop and resume without losing work. Commit the notebook
output as a dataset between sessions and add it as an input to the next one.

In [ ]:
!rm -rf /kaggle/working/tcwpn_test
!git clone -q https://github.com/dulhara79/tcwpn_test.git /kaggle/working/tcwpn_test
%cd /kaggle/working/tcwpn_test
!pip install -q -r requirements.txt 2>&1 | tail -2

import subprocess, sys, os
r = subprocess.run([sys.executable, "-m", "pytest",
                    "tests/test_repo_layout.py", "tests/test_call_arity.py",
                    "-q", "--no-header"],
                   capture_output=True, text=True,
                   env={**os.environ, "PYTHONPATH": "src"})
print(r.stdout[-2000:])
if r.returncode != 0:
    raise SystemExit("Repository layout is broken — fix before spending GPU time.")

In [ ]:
from pathlib import Path
STAGE_A_DS = Path("/kaggle/input/datasets/dulharakaushalya/tc-wpn-stage-a-data")
STAGE_A = next((c for c in (STAGE_A_DS/"data"/"clean", STAGE_A_DS/"clean", STAGE_A_DS)
                if (c/"pkl").exists()), None)
if STAGE_A is None:
    raise SystemExit(f"no pkl/ under {STAGE_A_DS}")

PKL_DIR, PLAN_DIR = "/kaggle/working/pkl", str(STAGE_A/"plans")
STEM, K, RESULTS = "psych_mimic4idx", 5, "/kaggle/working/results"
SEEDS   = [42, 43, 44, 45, 46]
CONFIGS = ["aux_only", "temporal_aux", "pcw_aux", "tcwpn_full"]
!mkdir -p {PKL_DIR}
!cp {STAGE_A}/pkl/*.pkl {PKL_DIR}/
print("ready")

## Recover whatever already exists — this is the Phase 2 fix

Phase 2's DeLong stage skipped every comparison because it looked only for
`best.pt`, and the older checkpoints were not among the notebook's inputs.

The important realisation: **DeLong does not need a checkpoint.** It needs the
paired score vectors, which live in `predictions_test.csv`. So this cell looks
for that file first and only falls back to a checkpoint when it is absent.

A likely reason the older files were unreachable at all: your `.gitignore`
excludes `*.csv` and `*.json`, so `predictions_test.csv` was never committed to
GitHub even though it existed in the Kaggle session. Add your **Stage C** and
**Phase 1** result datasets as inputs to this notebook. The cell prints exactly
what it found so you can tell recovery from silent absence.

In [ ]:
import os, glob, shutil

def recover(cfg, seed):
    """Return 'have_predictions' | 'have_checkpoint' | 'missing'."""
    name = f"{cfg}_k{K}_seed{seed}"
    dst  = f"{RESULTS}/{STEM}/{name}"
    if os.path.exists(f"{dst}/predictions_test.csv"):
        return "have_predictions"

    # 1. predictions CSV anywhere in the inputs -> no GPU work needed at all
    hits = glob.glob(f"/kaggle/input/**/{name}/predictions_test.csv", recursive=True)
    if hits:
        src = os.path.dirname(sorted(hits)[0])
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            if os.path.isfile(os.path.join(src, f)):
                shutil.copy2(os.path.join(src, f), dst)
        return "have_predictions"

    # 2. otherwise a checkpoint we can re-evaluate
    if os.path.exists(f"{dst}/best.pt"):
        return "have_checkpoint"
    hits = glob.glob(f"/kaggle/input/**/{name}/best.pt", recursive=True)
    if hits:
        src = os.path.dirname(sorted(hits)[0])
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            if os.path.isfile(os.path.join(src, f)):
                shutil.copy2(os.path.join(src, f), dst)
        return "have_checkpoint"
    return "missing"

status = {}
for cfg in CONFIGS:
    for seed in SEEDS:
        status[(cfg, seed)] = recover(cfg, seed)

import pandas as pd
grid = pd.DataFrame(
    [[status[(c, s)] for s in SEEDS] for c in CONFIGS],
    index=CONFIGS, columns=[f"seed{s}" for s in SEEDS])
print(grid.to_string())
print()
n_train = sum(v == "missing" for v in status.values())
print(f"to train this session : {n_train}  (~{n_train*48/60:.1f} h)")
print(f"to re-evaluate only   : {sum(v=='have_checkpoint' for v in status.values())}")
print(f"already complete      : {sum(v=='have_predictions' for v in status.values())}")

## Train what is missing

Skips anything already recovered. Safe to interrupt: rerun the notebook and it
picks up where it stopped.

In [ ]:
for cfg in CONFIGS:
    for seed in SEEDS:
        if status[(cfg, seed)] != "missing":
            continue
        print("="*70, f"\nTRAIN {cfg} seed={seed}\n", "="*70, sep="")
        !python -m scripts.train --config configs/{cfg}.yaml \
            --k {K} --seed {seed} --stem {STEM} \
            --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --results {RESULTS}

In [ ]:
# Evaluate everything that has a checkpoint but no predictions yet.
for cfg in CONFIGS:
    for seed in SEEDS:
        run = f"{RESULTS}/{STEM}/{cfg}_k{K}_seed{seed}"
        if os.path.exists(f"{run}/predictions_test.csv"):
            continue
        if not os.path.exists(f"{run}/best.pt"):
            continue
        !python -m scripts.evaluate --run {run} --split test \
            --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000

## Mean ± SD across seeds

The gate at the bottom of this output is the one that decides your paper. If the
between-configuration spread is smaller than the between-seed SD, the mechanisms
are not distinguishable and the paper reports them as equivalent.

In [ ]:
!python -m scripts.collect_seeds \
    --results {RESULTS}/{STEM} \
    --configs aux_only temporal_aux pcw_aux tcwpn_full \
    --seeds 42 43 44 45 46 --k {K} \
    --out-dir /kaggle/working/summary

## The completed DeLong ladder

Three comparisons, each isolating one mechanism with the auxiliary head held
constant, run on seed 42 so the pairing is exact. Holm correction, step-down.

In [ ]:
import subprocess, json
import pandas as pd

SEED_FOR_PAIRS = 42
PAIRS = [
    ("aux_only", "temporal_aux", "adds w^T  -> delta_temporal"),
    ("aux_only", "pcw_aux",      "adds w^C  -> delta_PCW"),
    ("aux_only", "tcwpn_full",   "adds both -> delta_interaction"),
]

out = []
for a, b, why in PAIRS:
    pa = f"{RESULTS}/{STEM}/{a}_k{K}_seed{SEED_FOR_PAIRS}/predictions_test.csv"
    pb = f"{RESULTS}/{STEM}/{b}_k{K}_seed{SEED_FOR_PAIRS}/predictions_test.csv"
    if not (os.path.exists(pa) and os.path.exists(pb)):
        print(f"skip {a} vs {b}: missing predictions"); continue
    dst = f"/kaggle/working/delong_{a}_vs_{b}.json"
    subprocess.run(["python", "-m", "scripts.compare_models", "pair",
                    "--a", pa, "--b", pb, "--out", dst], check=False)
    try:
        d = json.load(open(dst)); d["comparison"] = f"{a} -> {b}"; d["isolates"] = why
        out.append(d)
    except Exception as e:
        print("failed", a, b, e)

if out:
    t = pd.DataFrame(out).sort_values("p_value").reset_index(drop=True)
    m = len(t)
    t["holm_threshold"] = [0.05 / (m - i) for i in range(m)]
    t["significant"] = t["p_value"] <= t["holm_threshold"]
    fails = t.index[~t["significant"]]
    if len(fails):
        t.loc[fails.min():, "significant"] = False    # step-down
    cols = ["comparison", "isolates", "auroc_a", "auroc_b", "delta_auroc",
            "p_value", "holm_threshold", "significant"]
    print(t[[c for c in cols if c in t.columns]].to_string(index=False))
    t.to_csv("/kaggle/working/phase3_delong_holm.csv", index=False)
else:
    print("No comparisons ran. Check the recovery grid above.")

## Blinded robustness on the final configurations

Same patients, same episode plans, only the characters differ. Report retention
against the **above-chance margin**:

    retention = (AUROC_blind - 0.5) / (AUROC_original - 0.5)

Your Stage C numbers gave 0.7379 → 0.6284 for tcwpn_full under `anx_meds`,
which is 0.2379 → 0.1284 above chance: about 54% retained, so roughly **46% of
the signal was lexical**. Recompute for whichever configuration you finalise.

In [ ]:
for cfg in CONFIGS:
    run = f"{RESULTS}/{STEM}/{cfg}_k{K}_seed{SEED_FOR_PAIRS}"
    if not os.path.exists(f"{run}/best.pt"):
        print(f"skip {cfg}: no checkpoint in this session (blinding needs the model)")
        continue
    for level in ["anxiety", "anx_meds"]:
        !python -m scripts.evaluate --run {run} --split test --blind {level} \
            --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} --bootstrap 2000

In [ ]:
import json, glob, os
import pandas as pd

rows = []
for cfg in CONFIGS:
    run = f"{RESULTS}/{STEM}/{cfg}_k{K}_seed{SEED_FOR_PAIRS}"
    vals = {}
    for tag, label in [("test", "original"),
                       ("test_blind-anxiety", "anxiety_blind"),
                       ("test_blind-anx_meds", "anx_meds_blind")]:
        f = f"{run}/eval_{tag}.json"
        if os.path.exists(f):
            vals[label] = json.load(open(f))["metrics"]["auroc"]
    if "original" in vals:
        row = {"model": cfg, **{k: round(v, 4) for k, v in vals.items()}}
        base = vals["original"] - 0.5
        for label in ("anxiety_blind", "anx_meds_blind"):
            if label in vals and base > 0:
                row[f"margin_retained_{label}"] = round((vals[label]-0.5)/base, 3)
        rows.append(row)

if rows:
    r = pd.DataFrame(rows).set_index("model")
    print(r.to_string())
    r.to_csv("/kaggle/working/phase3_robustness.csv")
else:
    print("no blinded evaluations available yet")

## Writing it up

Send your supervisor `summary/seed_summary.csv`, `phase3_delong_holm.csv` and
`phase3_robustness.csv`.

If the seed SD swamps the configuration spread — which Phase 2 suggests it will —
then the honest paper is the one your supervisor already sketched in section 23:

> Standard episodic prototypical training collapses on a leakage-controlled
> anxiety-versus-psychiatric-control benchmark (prototype cosine 0.99998,
> score SD 0.00016). An auxiliary supervised objective rescues the episodic
> objective from that degenerate optimum, lowering the prototypical loss from
> ~0.69 to ~0.49 and raising AUROC from ~0.50 to ~0.74. Temporal recency and
> prototype-consistency weighting produce no measurable additional benefit
> under auxiliary-controlled ablation across five seeds.

Three things to carry into the text regardless of the outcome:

1. **Call the task concurrent detection at or before the index admission**, not
   forecasting.
2. **Report the index-time attrition**: 85.85% of cases retained versus 74.99%
   of controls, a 10.86-point differential, as a selection limitation.
3. **Report specificity**, which sits near 0.35–0.39. F1 alone is misleading at
   59.4% prevalence, where predicting everyone positive already scores 0.745.

Do not run a hyperparameter search on the auxiliary weight until this table
exists. Your supervisor's section 12 rules that out, and it is the single
easiest way to lose the credibility the leakage certificate bought you.